In [0]:
WITH customer_orders AS (
    SELECT
        customer_unique_id,
        COUNT(DISTINCT order_id) AS total_orders
    FROM workspace.default.final_quick_comm_dataset
    GROUP BY 1
),

repeat_customers AS (
    SELECT
        customer_unique_id
    FROM customer_orders
    WHERE total_orders > 1
),

order_level AS (
    SELECT
        order_id,
        customer_unique_id,
        product_category_name_english,
        MAX(payment_value) AS order_value
    FROM workspace.default.final_quick_comm_dataset
    GROUP BY 1,2,3
),
repeat_purchase AS (
    SELECT
        product_category_name_english,
        ROUND(SUM(
                CASE WHEN customer_unique_id IN (SELECT customer_unique_id FROM repeat_customers)
                     THEN order_value ELSE 0
                END), 2) AS repeat_customer_revenue,
        ROUND(SUM(
                CASE
                    WHEN customer_unique_id IN (SELECT customer_unique_id FROM repeat_customers) 
                    THEN order_value
                    ELSE 0 END) * 100.0/SUM(order_value),2) AS repeat_purchase_contribution_pct
    FROM order_level
    GROUP BY 1
)

SELECT
  a.product_category_name_english as category,
  COUNT(DISTINCT a.order_id) AS total_orders,
  ROUND(SUM(a.payment_value),2) AS total_revenue,
  ROUND(AVG(a.review_score),2) AS avg_review_score,
  ROUND(COUNT( CASE WHEN a.order_status = 'canceled' THEN a.order_id END)* 100.0 / COUNT(DISTINCT a.order_id),2) AS cancellation_rate,
  COUNT(CASE WHEN a.order_status = 'canceled' THEN a.order_id END) AS cancelled_orders,
  COUNT(DISTINCT CASE WHEN a.order_delivered_customer_date > a.order_estimated_delivery_date THEN a.order_id END) AS late_orders,
  ROUND(
        COUNT(DISTINCT CASE WHEN a.order_delivered_customer_date > a.order_estimated_delivery_date THEN a.order_id END)* 100.0
        / COUNT(DISTINCT a.order_id), 2) AS late_delivery_rate,
  r.repeat_customer_revenue,
  r.repeat_purchase_contribution_pct
FROM workspace.default.final_quick_comm_dataset a  
JOIN repeat_purchase r ON a.product_category_name_english = r.product_category_name_english
GROUP BY a.product_category_name_english, r.repeat_customer_revenue, r.repeat_purchase_contribution_pct;